In [4]:
import pandas as pd
import sys
sys.path.append("../")
from config.config import RagConfig
from build_knowledge_vectorstore import get_bm25_retriever, generation_replay_by_LLM

In [ ]:
qa_raw_data=pd.read_csv("../data/金融QA数据集.csv", encoding="GB2312")
print(qa_raw_data)

print(qa_raw_data["Question"])

#导入文档型知识库
doc_raw_data = pd.read_csv("../data/文档型知识.txt")
doc_raw_data

In [ ]:
query= "：高利率时代的全球博弈：2024年金融时政变局与结构性挑战"
bm25_retriever = get_bm25_retriever(K=10)
bm25_result = bm25_retriever.invoke(query)
bm25_result

documents: [Document(metadata={'source': '../data/文档型知识.txt'}, page_content='标题：高利率时代的全球博弈：2024年金融时政变局与结构性挑战\n一、宏观背景：三重裂变下的世界经济格局\n货币政策的分化与共振\n美联储维持5.25%-5.5%利率区间已达12个月，欧元区跟进紧缩，而日本央行结束负利率时代却步履蹒跚。这种分化导致套利资本剧烈流动，新兴市场货币承压（如印尼盾年内贬值6.2%）。\n关键矛盾：美国6月CPI回落至3%但核心通胀黏性仍存，市场对降息预期的反复修正引发美债收益率震荡（10年期收益率4.2%→3.8%→4.1%）。\n地缘政治的经济成本\n俄乌冲突进入第三年，欧洲能源转型代价显现：德国工业用电价格同比上涨47%，巴斯夫等化工巨头加速海外产能转移。\n红海危机推升全球物流成本（上海至欧洲集装箱运价翻倍），叠加巴以冲突，地缘风险溢价持续注入原油市场（布伦特原油85→92美元/桶）。\n债务雪球的膨胀\nIMF数据显示，全球债务总额达307万亿美元（占GDP336%），美国政府债务规模突破34万亿美元，年化利息支付成本超1万亿美元。债务货币化与去美元化进程形成对冲。\n二、结构性矛盾深度解析\n（1）美国：经济韧性背后的政治经济学\n数据悖论：Q2 GDP增速2.4%但贫富分化加剧（基尼系数0.49），"拜登经济学"的产业政策（《芯片法案》527亿美元）与贸易保护主义（对中国电动车关税提至100%）形成撕裂。\n选举周期扰动：特朗普若胜选可能引发：① 2017年减税政策延期；② 对华关税全面升级；③ 美联储独立性面临挑战。\n（2）欧洲：战略自主性的困局\n欧盟碳关税（CBAM）正式实施暴露产业短板：2023年对华光伏组件进口反增62%，绿色转型被迫依赖中国供应链。\n法国财政赤字率4.9%突破欧盟红线，德国宪法法院裁定600亿欧元气候基金违宪，财政碎片化制约复苏。\n（3）新兴市场：脆弱的平衡术\n印度借"中国+1"战略吸引外资（苹果供应链转移创造15万岗位），但卢比国际化进程受阻（跨境结算占比仅1.6%）。\n沙特"2030愿景"遭遇油价波动，NEOM新城建设进度滞后，主权财富基金减持特斯拉等流动性资产应对财政压力。\n三、市场信号与政策困境\n债券市场的警示\n美国2/10

[Document(metadata={'type': 'bm25'}, page_content='标题：高利率时代的全球博弈：2024年金融时政变局与结构性挑战'),
 Document(metadata={'type': 'bm25'}, page_content='结语：重构全球治理的紧迫性'),
 Document(metadata={'type': 'bm25'}, page_content='数字货币的政经博弈'),
 Document(metadata={'type': 'bm25'}, page_content='什么是金融衍生品？答案： '),
 Document(metadata={'type': 'bm25'}, page_content='\n灰犀牛风险：① 美国商业地产贷款违约潮（2024年到期债务达5600亿美元）'),
 Document(metadata={'type': 'bm25'}, page_content='货币政策的分化与共振'),
 Document(metadata={'type': 'bm25'}, page_content='产业政策竞赛的全球回响'),
 Document(metadata={'type': 'bm25'}, page_content='\n选举周期扰动：特朗普若胜选可能引发：① 2017年减税政策延期'),
 Document(metadata={'type': 'bm25'}, page_content='什么是债务与权益比率？答案：运'),
 Document(metadata={'type': 'bm25'}, page_content='二、结构性矛盾深度解析')]

In [ ]:
import json
background_knowledge=("美联储维持5.25%-5.5%利率区间已达12个月，欧元区跟进紧缩，而日本央行结束负利率时代却步履蹒跚。" #知识1
                     "市场对降息预期的反复修正引发美债收益率震荡（10年期收益率4.2%→3.8%→4.1%）") #知识2
history_message="""用户:你好\nAI:你好，有什么能帮到您的？"""
current_query="美联储维持的利率引发欧元和美债的影响是什么？"
try:
    response = generation_replay_by_LLM(background_knowledge,history_message,current_query)
    print(response)
except Exception as e:
        print(e)

verbose=False prompt=PromptTemplate(input_variables=['background_knowledge', 'current_query', 'history_message'], input_types={}, partial_variables={}, template='你是一个人工客服,名叫小丽,你能用礼貌的态度，基于背景信息回答用户的问题。如果背景信息与用户的问题相关则需要结合背景信息进行回答(不能乱编)，\n        如果背景信息与用户的问题无关则忽略背景信息，直接回答。\n        背景信息:{background_knowledge}\n        历史记录:{history_message}\n        用户问题:{current_query}\n        回答：') llm=OpenAI(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.11'}}, client=<openai.resources.completions.Completions object at 0x77094e868c20>, async_client=<openai.resources.completions.AsyncCompletions object at 0x77094e868d70>, model_name='deepseek-ai/DeepSeek-V3.2', top_p=0.9, model_kwargs={}, openai_api_key='sk-sfgmvtrouzaciiedntxqtrirwbjintgdonatpxclqludixye', openai_api_base='https://api.siliconflow.cn/v1', openai_proxy='', logit_bias={}) output_parser=StrOutputParser() llm_kwargs={}
the JSON object must be str, bytes or bytearray, not dict


In [ ]:
import requests
from config.config import RagConfig
demo_url = f"http://{RagConfig().server_host}:7069/rag_chat"
query="股票期权给我介绍下是什么" #股票期权是一种赋予持有者在未来某一日期以特定价格买卖股票的权利的金融衍生品。
# query="你好"
history=[{"role":"user","content":"你好"},{"role":"assistant","content":"请问有什么能帮到您的呢？"}]
_data = {
    "query":query, #平顶山 河南 你好
    "history":history
   }
test_response = requests.post(demo_url, json=_data,timeout=30)
print("test_response",test_response)
test_dict=test_response.json()
print(test_dict)